# A pendulum

A point mass on a massless rod, hinged at one end: the smallest model with something to check. This notebook builds it, swings it, and holds each result against its closed form:

1. the period of a large swing, from the complete elliptic integral;
2. the speed at the bottom and the pull in the rod, from the energy;
3. the period at every amplitude, against the small-angle period;
4. how three integrators keep the energy;
5. the small oscillations, from the linearised model.

Then it puts the swing in a table and plays it back in Rerun.

Each check prints a result beside its closed form and stops the notebook if they differ by more than the allowance derived beside it. `python/tests/test_notebooks.py` runs the notebook in the same way, so these numbers are tested. (Plan task 6.6.)

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

import multibody as mbd


def check(what, value, expected, allowed):
    """Print a result beside the value it should have; stop if they differ by more than `allowed`."""
    difference = float(np.max(np.abs(np.asarray(value) - np.asarray(expected))))
    if np.ndim(value) == 0:
        print(f"{what}: {value:.12g}, expected {expected:.12g}, difference {difference:.1e} (allowed {allowed:.1e})")
    else:
        print(f"{what}: largest difference {difference:.1e} (allowed {allowed:.1e})")
    assert difference <= allowed, what

## The model

A bob of 1 kg on a rod of 0.8 m. The hinge turns about Z and gravity is −Y, so the bob swings in the XY plane. The bob's frame is at the mass, and its joint frame is 0.8 m above it (`x_cj`). So at q = 0 the bob hangs straight down, and q is the angle from hanging.

In [ ]:
L, m, g = 0.8, 1.0, mbd.g_accel


def pendulum():
    system = mbd.System()
    system.model.gravity = [0.0, -g, 0.0]
    system.model.add_body("ground", mbd.RevoluteJointModel(), x_cj=[0.0, L, 0.0],
                          inertia=mbd.RigidBodyInertia(m, np.zeros(3), np.zeros((3, 3))), name="bob")
    return system


print(mbd.describe(pendulum()))

## A large swing

Released at rest from 1 rad (57°). Dormand-Prince, with its tolerances at 1e-12, keeps its error far below what is checked. An event watches q: at each falling crossing of zero, as the bob passes the bottom moving towards negative q, it notes the time, the angle and the rate (the grey lines).

In [ ]:
theta0 = 1.0


def swing(theta0, duration, passes=None):
    """Release the pendulum at rest from theta0 and run it for `duration`, sampled every 10 ms. Each pass
    through the bottom towards negative q is appended to `passes` as (time, angle, rate)."""
    sim = mbd.Simulator(pendulum())
    sim.method = mbd.Integrator.DormandPrince
    sim.step_control.relative_tolerance = 1e-12
    sim.step_control.absolute_tolerance = 1e-12
    if passes is not None:
        sim.add_event("bottom", lambda s: s.q[0], lambda s: passes.append((s.time, s.q[0], s.v[0])), direction=-1)
    sim.q[0] = theta0
    sim.initialize()
    return sim, sim.run_recorded(duration, 0.01)


passes = []
sim, run = swing(theta0, 4.0, passes)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(run["time"], run["q"][:, 0], label="angle q [rad]")
ax.plot(run["time"], run["v"][:, 0], label="rate dq/dt [rad/s]")
for t, _, _ in passes:
    ax.axvline(t, color="grey", lw=0.8)
ax.set_xlabel("time [s]")
ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False)
ax.grid(True)

### The period

Released at rest, the bob comes back to the same state once a period T, so passes through the bottom in the same direction are T apart. By symmetry the first comes a quarter period after the release. The closed form is

$$T = 4\sqrt{L/g}\;K(k), \qquad k = \sin(\theta_0/2),$$

with K the complete elliptic integral of the first kind, computed here by the arithmetic-geometric mean (Abramowitz and Stegun 17.6). The event places a crossing within 1e-10 s, and the integrator's error at its tolerances is smaller still: allowed 1e-9 s.

In [ ]:
def K(k):
    """The complete elliptic integral of the first kind, pi / (2 AGM(1, sqrt(1 - k^2)))."""
    a, b = 1.0, math.sqrt(1.0 - k * k)
    while abs(a - b) > 1e-15 * a:
        a, b = 0.5 * (a + b), math.sqrt(a * b)
    return math.pi / (2.0 * a)


def period(theta0):
    return 4.0 * math.sqrt(L / g) * K(math.sin(0.5 * theta0))


check("period [s]", passes[1][0] - passes[0][0], period(theta0), 1e-9)
check("first pass, a quarter period [s]", passes[0][0], period(theta0) / 4.0, 1e-9)
print(f"(the small-angle period, 2 pi sqrt(L/g), is {2.0 * math.pi * math.sqrt(L / g):.6f} s)")

### The speed at the bottom and the pull in the rod

Energy is conserved. Having fallen L(1 − cos θ0), the bob passes the bottom at the rate θ̇ = √(2g(1 − cos θ0)/L). There the rod both holds the weight and turns the bob on its circle, so it pulls with m g + m L θ̇² = m g (3 − 2 cos θ0).

`compute_loads` gives what the hinge passes to the bob, [moment; force] in the joint frame, which at q = 0 is aligned with the world. The integrator's tolerance, 1e-12 per step over the 110 steps to the bottom, bounds the rate's error by about 4e-10 rad/s: allowed 1e-9 rad/s. The pull then errs by at most 2 m L θ̇ × 1e-9, 6e-9 N: allowed 1e-8 N.

In [ ]:
_, angle, rate = passes[0]
check("rate at the bottom [rad/s]", -rate, math.sqrt(2.0 * g * (1.0 - math.cos(theta0)) / L), 1e-9)
at_bottom = mbd.Simulator(pendulum())
at_bottom.q[0], at_bottom.v[0] = angle, rate
at_bottom.refresh()
reaction = mbd.compute_loads(at_bottom).joint_reaction[1]      # [moment; force] on the bob, joint frame
check("pull of the rod [N]", reaction[4], m * g * (3.0 - 2.0 * math.cos(theta0)), 1e-8)

## The period at every amplitude

The same release from fifteen angles between 0.1 rad and 3.0 rad (172°, nearly upside down); each period is four times the time to the first pass. Near the top the bob lingers, and the period grows without bound as θ0 approaches π. The small-angle period 2π√(L/g) holds only at the left of the plot. Allowed 1e-9 s each, as above.

In [ ]:
amplitudes = np.linspace(0.1, 3.0, 15)
simulated = []
for amplitude in amplitudes:
    first = []
    swing(amplitude, 0.3 * period(amplitude), first)
    simulated.append(4.0 * first[0][0])
check("periods [s]", simulated, [period(amplitude) for amplitude in amplitudes], 1e-9)

small = 2.0 * math.pi * math.sqrt(L / g)
fine = np.linspace(0.01, 3.05, 300)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(fine, [period(a) / small for a in fine], label=r"closed form, $4\sqrt{L/g}\,K(\sin(\theta_0/2))$")
ax.plot(amplitudes, np.array(simulated) / small, "o", label="simulated")
ax.axhline(1.0, color="grey", ls="--", label=r"small angles, $2\pi\sqrt{L/g}$")
ax.set_xlabel(r"released from $\theta_0$ [rad]")
ax.set_ylabel("period / small-angle period")
ax.legend()
ax.grid(True)

## How three integrators keep the energy

Nothing dissipates energy here, so E = ½ m L² θ̇² − m g L cos θ should keep its initial value. Three integrators, each run for 20 s:

- **RK4**, with a fixed step of 10 ms. Its error is of fourth order, but nothing in it conserves energy: E drifts, slowly and steadily.
- **Semi-implicit Euler**, with a fixed step h of 10 ms. It is only first order, so its energy error is far larger, but it is symplectic: it keeps exactly a nearby energy, the modified Hamiltonian $\tilde E = E - \tfrac{h}{2}\,\dot\theta\,V'(\theta) + O(h^2)$, with V′ = m g L sin θ (Hairer, Lubich and Wanner, *Geometric Numerical Integration*, IX.1). So E swings but does not drift, and its largest swing is (h/2) max|θ̇ V′|, to within the terms of order h². Those are below (h²/12)(m g² + m g L max θ̇²) ≈ 1.5e-3 J at each end: allowed 3e-3 J for both checks.
- **Dormand-Prince**, with adaptive steps and tolerances of 1e-10. Its error stays at the level of its tolerance.

These runs have no event: an event cuts the step at its crossing, and a semi-implicit Euler with uneven steps is no longer symplectic.

In [ ]:
def energy_run(method, step=0.01):
    sim = mbd.Simulator(pendulum())
    sim.method = method
    sim.step_control.relative_tolerance = 1e-10
    sim.step_control.absolute_tolerance = 1e-10
    sim.q[0] = theta0
    sim.initialize()
    run = sim.run_recorded(20.0, step)
    q, v = run["q"][:, 0], run["v"][:, 0]
    return run["time"], q, v, 0.5 * m * L * L * v * v - m * g * L * np.cos(q)


h = 0.01
fig, ax = plt.subplots(figsize=(8, 3.5))
for method in (mbd.Integrator.SemiImplicitEuler, mbd.Integrator.RK4, mbd.Integrator.DormandPrince):
    time, q, v, E = energy_run(method, h)
    ax.semilogy(time[1:], np.abs(E[1:] - E[0]), lw=0.8, label=method.name)
    print(f"{method.name}: largest |E - E0| {np.max(np.abs(E - E[0])):.1e} J")
    if method == mbd.Integrator.SemiImplicitEuler:
        swing_term = 0.5 * h * np.abs(v * m * g * L * np.sin(q))
        check("semi-implicit Euler: largest |E - E0| [J]", np.max(np.abs(E - E[0])), np.max(swing_term), 3e-3)
        modified = E - 0.5 * h * v * m * g * L * np.sin(q)
        check("semi-implicit Euler: change of the modified energy [J]", np.ptp(modified), 0.0, 3e-3)
ax.set_xlabel("time [s]")
ax.set_ylabel("|E - E0| [J]")
ax.legend(loc="lower right")
ax.grid(True, which="both", lw=0.3)

## Small oscillations

`linearize` gives the linear model about the state the simulator holds, here hanging at rest: one mode, of frequency √(g/L), undamped. It differentiates the equations of motion by central differences with a perturbation of 1e-6, whose truncation error, (1e-6)² times the third derivative, is about 1e-12, and whose rounding error is about 1e-16 / 1e-6 = 1e-10 relative: allowed 1e-8 relative.

In [ ]:
linear = mbd.linearize(mbd.Simulator(pendulum()))
print(linear)
check("natural frequency [rad/s]", linear.modes[0].natural_frequency, math.sqrt(g / L), 1e-8 * math.sqrt(g / L))
check("damping ratio", linear.modes[0].damping_ratio, 0.0, 1e-8)

## The swing as a table

`mbd.dataframe` turns a run into a pandas table indexed by time, its columns named after the bodies (`q` and `v`, then the body). `mbd.write_results(table, "swing.parquet")` would save it.

In [ ]:
table = mbd.dataframe(run, sim)
table.head()

## Playing it back

`mbd.replay` opens the Rerun viewer with the large swing. The bob, which has no size, is drawn as a point, with a line to its hinge; q and its rate are plotted beside it. Drag the time line, or press play. Each call makes a new recording, listed under the name given, so running the cell again adds a second one. With `path="pendulum.rrd"` the recording is saved to a file instead.

In [ ]:
mbd.replay(sim.system, run, name="pendulum");